# Main code

In [ ]:
# %% [markdown]
# # Experiment C Revised
# ## Proxy count k vs latent dimension d
#
# This notebook revises the IntervalGP-VAE in the first code to be consistent
# with the IntervalGP-VAE framework and parameters in the second code.
#
# Main features:
#
# 1. Keeps the first code's Experiment C purpose:
#    - Test proxy count k vs latent dimension d
#    - Evaluate the scaling condition k >= 2d + 4
#
# 2. Uses the second code's IntervalGP-VAE framework:
#    - chosen_version = "u_aux"
#    - DoubleEncoder with latent variables u, eps, ua0, ua1
#    - AdditiveDecoder with auxiliary latents
#    - FlexibleCausalHead using u, t, and ua1
#    - local IntervalGP regularizer during training
#    - latent GP posterior smoothing at inference
#    - ITE-space GP posterior interval at inference
#    - three-stage training:
#        Stage 1: joint VAE + causal head
#        Stage 2: frozen VAE, train causal head
#        Stage 3: frozen causal head, refine VAE
#
# 3. Larger sample size for larger latent dimension d:
#    - d=1: N_train=1200, N_test=400
#    - d=2: N_train=1600, N_test=500
#    - d=3: N_train=2200, N_test=600
#    - d=4: N_train=3000, N_test=800
#    - d=5: N_train=4000, N_test=1000


# %%
# ============================================================
# Cell 1: Imports
# ============================================================

import os
import gc
import time
import random
import warnings
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")


# %%
# ============================================================
# Cell 2: Global settings
# ============================================================

OUTPUT_DIR = "experiment_c_consistent_intervalgpvae_larger_sample_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# The second code uses CPU.
# You may change this to "cuda" if available and stable.
DEVICE = "cpu"

GLOBAL_SEED = 420

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(GLOBAL_SEED)

torch.set_default_dtype(torch.float32)


# ============================================================
# Larger sample size for larger d
# ============================================================

N_TRAIN_BY_D = {
    1: 1200,
    2: 1600,
    3: 2200,
    4: 3000,
    5: 4000,
}

N_TEST_BY_D = {
    1: 400,
    2: 500,
    3: 600,
    4: 800,
    5: 1000,
}


# ============================================================
# Experiment grid
# ============================================================

D_LIST = [1, 2, 3, 4, 5]

# K_GRID_BY_D = {
#     1: [3, 4, 6, 8, 10],
#     2: [4, 6, 8, 10, 12],
#     3: [5, 8, 10, 12, 15],
#     4: [6, 8, 10, 12, 14, 16],
#     5: [7, 10, 12, 14, 16, 20],
# }

K_GRID_BY_D = {
    1: list(range(3, 10)),   # 1,2,3,4,5,6,7,8,9,10
    2: list(range(4, 12)),   # 4,5,6,7,8,9,10,11,12
    3: list(range(5, 14)),   # 5,6,7,8,9,10,11,12,13,14,15
    4: list(range(6, 16)),   # 6,7,8,9,10,11,12,13,14,15,16
    5: list(range(7, 18)),   # 7,8,9,10,11,12,13,14,15,16,17,18,19,20
}

SEEDS = [11, 22, 33, 44, 55]


# ============================================================
# Data-generation settings
# ============================================================

NOISE_Z = 0.10
NOISE_Y = 0.10


# ============================================================
# IntervalGP-VAE parameters consistent with the second code
# ============================================================

chosen_version = "u_aux"

INTERVALGPVAE_HIDDEN_DIM = 64
CAUSAL_HEAD_HIDDEN_DIM = 64

GP_LENGTHSCALE = 7.0
GP_VARIANCE = 2.0
GP_NOISE = 1e-4

BATCH_SIZE = 128

JOINT_EPOCHS = 200
HEAD_EPOCHS = 100
VAE_REFINE_EPOCHS = 50

LR_JOINT = 1e-3
LR_HEAD = 1e-3
LR_VAE_REFINE = 1e-4
WEIGHT_DECAY = 1e-5

ITE_CI = 0.90
Z_VALUE_90 = 1.6448536269514722
N_ITE_SAMPLES = 100

# ------------------------------------------------------------
# GP reference control
# ------------------------------------------------------------
# The second code uses MAX_GP_POINTS=None by default.
# However, because this experiment has many d,k,seed combinations and larger
# samples for larger d, we use a bounded GP reference set for computational
# realism.
#
# If you want exact full GP over all training samples, set:
#     MAX_GP_POINTS = None
#
# If runtime is too slow, use:
#     MAX_GP_POINTS = 200
MAX_GP_POINTS = 300


# ============================================================
# Plot control
# ============================================================

MAKE_HEATMAPS = True


# %%
# ============================================================
# Cell 3: Utility functions
# ============================================================

def to_numpy_1d(x):
    if torch.is_tensor(x):
        x = x.detach().cpu().numpy()
    return np.asarray(x).reshape(-1)


def safe_corr(x, y):
    x = to_numpy_1d(x)
    y = to_numpy_1d(y)

    valid = np.isfinite(x) & np.isfinite(y)

    if valid.sum() < 2:
        return np.nan

    if np.std(x[valid]) == 0 or np.std(y[valid]) == 0:
        return np.nan

    return float(np.corrcoef(x[valid], y[valid])[0, 1])


def pehe_np(est_ite, true_ite):
    est_ite = to_numpy_1d(est_ite)
    true_ite = to_numpy_1d(true_ite)

    valid = np.isfinite(est_ite) & np.isfinite(true_ite)

    if valid.sum() == 0:
        return np.nan

    return float(np.sqrt(np.mean((est_ite[valid] - true_ite[valid]) ** 2)))


def compute_interval_metrics(lower, upper, true_value):
    lower = to_numpy_1d(lower)
    upper = to_numpy_1d(upper)
    true_value = to_numpy_1d(true_value)

    valid = (
        np.isfinite(lower)
        & np.isfinite(upper)
        & np.isfinite(true_value)
    )

    if valid.sum() == 0:
        return {
            "coverage": np.nan,
            "avg_length": np.nan,
            "n": 0,
        }

    covered = (
        (lower[valid] <= true_value[valid])
        & (true_value[valid] <= upper[valid])
    )

    return {
        "coverage": float(np.mean(covered)),
        "avg_length": float(np.mean(upper[valid] - lower[valid])),
        "n": int(valid.sum()),
    }


def standardize_train_test(
    x_train: torch.Tensor,
    x_test: torch.Tensor,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    mean = x_train.mean(dim=0, keepdim=True)
    std = x_train.std(dim=0, keepdim=True).clamp_min(1e-6)

    return (
        (x_train - mean) / std,
        (x_test - mean) / std,
        mean,
        std,
    )


def latent_recovery_error_linear_aligned(u_true, u_hat):
    """
    Evaluate latent recovery after affine linear alignment.

    Because latent variables are identifiable only up to invertible
    transformation/sign/scale/rotation, direct RMSE between U and U_hat is
    not meaningful.
    """
    u_true = np.asarray(u_true, dtype=np.float64)
    u_hat = np.asarray(u_hat, dtype=np.float64)

    if u_true.ndim == 1:
        u_true = u_true.reshape(-1, 1)

    if u_hat.ndim == 1:
        u_hat = u_hat.reshape(-1, 1)

    u_true_std = (
        u_true - u_true.mean(axis=0, keepdims=True)
    ) / (u_true.std(axis=0, keepdims=True) + 1e-8)

    u_hat_std = (
        u_hat - u_hat.mean(axis=0, keepdims=True)
    ) / (u_hat.std(axis=0, keepdims=True) + 1e-8)

    X = np.concatenate(
        [u_hat_std, np.ones((u_hat_std.shape[0], 1))],
        axis=1,
    )

    B, *_ = np.linalg.lstsq(X, u_true_std, rcond=None)

    u_aligned = X @ B

    return float(np.sqrt(np.mean((u_aligned - u_true_std) ** 2)))


def latent_abs_corr_mean(u_true, u_hat):
    """
    For each true latent dimension, compute the best absolute correlation
    with any estimated latent dimension, and then average.
    """
    u_true = np.asarray(u_true)
    u_hat = np.asarray(u_hat)

    if u_true.ndim == 1:
        u_true = u_true.reshape(-1, 1)

    if u_hat.ndim == 1:
        u_hat = u_hat.reshape(-1, 1)

    corrs = np.zeros((u_true.shape[1], u_hat.shape[1]))

    for i in range(u_true.shape[1]):
        for j in range(u_hat.shape[1]):
            c = np.corrcoef(u_true[:, i], u_hat[:, j])[0, 1]
            corrs[i, j] = 0.0 if np.isnan(c) else abs(c)

    return float(corrs.max(axis=1).mean())


def maybe_subsample_gp_reference(x, *ys, max_points=None, seed=0):
    """
    Subsample GP reference points if max_points is not None.
    This still uses full GP posterior over the selected reference set.
    """
    n = x.shape[0]

    if max_points is None or n <= max_points:
        return (x, *ys)

    rng = np.random.default_rng(seed)
    idx = rng.choice(np.arange(n), size=max_points, replace=False)
    idx = np.sort(idx)

    idx_t = torch.tensor(idx, dtype=torch.long, device=x.device)

    out = [x[idx_t]]

    for y in ys:
        if torch.is_tensor(y):
            out.append(y[idx_t])
        else:
            out.append(np.asarray(y)[idx])

    return tuple(out)


def reset_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# %%
# ============================================================
# Cell 4: Synthetic multi-dimensional data generator
# ============================================================

@dataclass
class SyntheticData:
    z: torch.Tensor
    t: torch.Tensor
    y: torch.Tensor
    u: torch.Tensor
    y0: torch.Tensor
    y1: torch.Tensor
    ite: torch.Tensor
    proxy_directions: np.ndarray
    proxy_params: List[Dict[str, float]]
    treatment_direction: np.ndarray
    tau_direction: np.ndarray


def sigmoid_np(x: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-x))


def generate_diverse_directions_with_relevant_axes(
    k: int,
    d: int,
    rng: np.random.Generator,
    treatment_direction: Optional[np.ndarray] = None,
    tau_direction: Optional[np.ndarray] = None,
) -> np.ndarray:
    """
    Safe direction generator.

    This avoids infinite loops in low-dimensional settings, especially d=1.
    """
    directions = []

    if treatment_direction is not None and len(directions) < k:
        a_t = treatment_direction.astype(np.float32)
        a_t = a_t / (np.linalg.norm(a_t) + 1e-8)
        directions.append(a_t)

    if tau_direction is not None and len(directions) < k:
        a_tau = tau_direction.astype(np.float32)
        a_tau = a_tau / (np.linalg.norm(a_tau) + 1e-8)
        directions.append(a_tau)

    while len(directions) < k:
        a = rng.normal(0.0, 1.0, size=(d,)).astype(np.float32)
        a = a / (np.linalg.norm(a) + 1e-8)
        directions.append(a)

    directions = np.stack(directions[:k], axis=0).astype(np.float32)

    directions = directions / (
        np.linalg.norm(directions, axis=1, keepdims=True) + 1e-8
    )

    return directions


def generate_proxy_params(
    k: int,
    rng: np.random.Generator,
) -> List[Dict[str, float]]:
    """
    Randomized proxy basis parameters.
    """
    params = []

    for i in range(k):
        params.append({
            "kind": int(i % 8),
            "omega": float(rng.uniform(0.7, 2.0)),
            "phi": float(rng.uniform(-np.pi, np.pi)),
            "scale": float(rng.uniform(0.8, 1.5)),
            "identity_weight": float(rng.uniform(0.15, 0.35)),
        })

    return params


def proxy_transform_single(
    s: np.ndarray,
    param: Dict[str, float],
) -> np.ndarray:
    kind = int(param["kind"])
    omega = param["omega"]
    phi = param["phi"]
    scale = param["scale"]
    identity_weight = param["identity_weight"]

    s_scaled = scale * s

    if kind == 0:
        g = s

    elif kind == 1:
        g = np.sin(omega * s + phi) + identity_weight * s

    elif kind == 2:
        g = np.tanh(s_scaled) + identity_weight * s

    elif kind == 3:
        g = 0.5 * (s ** 2) + identity_weight * s

    elif kind == 4:
        g = np.exp(-0.2 * s_scaled ** 2) + identity_weight * s

    elif kind == 5:
        g = np.log1p(np.abs(s_scaled)) + identity_weight * s

    elif kind == 6:
        g = (
            np.sin(omega * s + phi)
            + 0.5 * np.cos(0.5 * omega * s)
            + identity_weight * s
        )

    else:
        g = (
            np.log1p(np.exp(np.clip(s_scaled, -20, 20)))
            + identity_weight * s
        )

    return g.astype(np.float32)


def make_proxy_features(
    u: np.ndarray,
    directions: np.ndarray,
    proxy_params: List[Dict[str, float]],
) -> np.ndarray:
    """
    Proxy construction:
        s_i = a_i^T U,
        Z_i = nonlinear_basis_i(s_i) + lambda_i s_i + epsilon_i.
    """
    k = directions.shape[0]

    assert len(proxy_params) == k, "proxy_params length must match k."

    z_list = []

    for i in range(k):
        a = directions[i]
        s = u @ a.reshape(-1, 1)

        g = proxy_transform_single(
            s=s,
            param=proxy_params[i],
        )

        z_list.append(g)

    return np.concatenate(z_list, axis=1).astype(np.float32)


def structural_h(u: np.ndarray) -> np.ndarray:
    """
    Baseline outcome h(U).
    """
    d = u.shape[1]

    weights = np.linspace(0.25, 0.75, d).reshape(-1, 1)

    linear = u @ weights

    nonlinear = 0.30 * np.sin(u[:, [0]])

    if d >= 2:
        nonlinear += 0.20 * np.tanh(u[:, [1]])

    if d >= 3:
        nonlinear += 0.10 * u[:, [2]] ** 2

    if d >= 4:
        nonlinear += 0.10 * np.sin(u[:, [3]] + 0.5 * u[:, [0]])

    if d >= 5:
        nonlinear += 0.08 * np.tanh(u[:, [4]] * u[:, [1]])

    return linear + nonlinear


def structural_tau(
    u: np.ndarray,
    tau_direction: np.ndarray,
) -> np.ndarray:
    """
    Individual treatment effect tau(U).
    """
    s = u @ tau_direction.reshape(-1, 1)

    tau = (
        0.70
        + 0.35 * np.tanh(s)
        + 0.15 * np.sin(u[:, [0]])
    )

    if u.shape[1] >= 2:
        tau += 0.10 * u[:, [1]]

    if u.shape[1] >= 3:
        tau += 0.05 * np.sin(u[:, [2]])

    if u.shape[1] >= 5:
        tau += 0.05 * np.tanh(u[:, [4]])

    return tau.astype(np.float32)


def generate_synthetic_multidim(
    n: int,
    d: int,
    k: int,
    seed: int,
    noise_z: float = NOISE_Z,
    noise_y: float = NOISE_Y,
    proxy_directions: Optional[np.ndarray] = None,
    proxy_params: Optional[List[Dict[str, float]]] = None,
    treatment_direction: Optional[np.ndarray] = None,
    tau_direction: Optional[np.ndarray] = None,
) -> SyntheticData:
    """
    Data-generating mechanism:
        U ~ N(0, I_d)
        Z_i = g_i(U) + eps_i
        T ~ Bernoulli(sigmoid(a_T^T U))
        Y(t) = h(U) + t * tau(U) + eps_y
        ITE = tau(U)
    """
    rng = np.random.default_rng(seed)

    u = rng.normal(0.0, 1.0, size=(n, d)).astype(np.float32)

    if treatment_direction is None:
        treatment_direction = rng.normal(0.0, 1.0, size=(d,)).astype(np.float32)
        treatment_direction = treatment_direction / (
            np.linalg.norm(treatment_direction) + 1e-8
        )

    if tau_direction is None:
        tau_direction = rng.normal(0.0, 1.0, size=(d,)).astype(np.float32)
        tau_direction = tau_direction / (
            np.linalg.norm(tau_direction) + 1e-8
        )

    if proxy_directions is None:
        proxy_directions = generate_diverse_directions_with_relevant_axes(
            k=k,
            d=d,
            rng=rng,
            treatment_direction=treatment_direction,
            tau_direction=tau_direction,
        )

    if proxy_params is None:
        proxy_params = generate_proxy_params(
            k=k,
            rng=rng,
        )

    clean_z = make_proxy_features(
        u=u,
        directions=proxy_directions,
        proxy_params=proxy_params,
    )

    z = clean_z + rng.normal(
        0.0,
        noise_z,
        size=clean_z.shape,
    ).astype(np.float32)

    logits = 1.0 * (u @ treatment_direction.reshape(-1, 1))
    p_t = sigmoid_np(logits).reshape(-1)

    t = rng.binomial(1, p_t).astype(np.float32)

    h_u = structural_h(u)
    tau_u = structural_tau(u, tau_direction)

    eps_y0 = rng.normal(0.0, noise_y, size=(n, 1)).astype(np.float32)
    eps_y1 = rng.normal(0.0, noise_y, size=(n, 1)).astype(np.float32)

    y0 = h_u + eps_y0
    y1 = h_u + tau_u + eps_y1

    y_obs = (
        h_u
        + t.reshape(-1, 1) * tau_u
        + rng.normal(0.0, noise_y, size=(n, 1)).astype(np.float32)
    )

    return SyntheticData(
        z=torch.tensor(z, dtype=torch.float32),
        t=torch.tensor(t, dtype=torch.float32),
        y=torch.tensor(y_obs.squeeze(), dtype=torch.float32),
        u=torch.tensor(u, dtype=torch.float32),
        y0=torch.tensor(y0.squeeze(), dtype=torch.float32),
        y1=torch.tensor(y1.squeeze(), dtype=torch.float32),
        ite=torch.tensor(tau_u.squeeze(), dtype=torch.float32),
        proxy_directions=proxy_directions,
        proxy_params=proxy_params,
        treatment_direction=treatment_direction,
        tau_direction=tau_direction,
    )


# %%
# ============================================================
# Cell 5: GP posterior functions
# ============================================================

def rbf_kernel(
    x1,
    x2=None,
    lengthscale=1.0,
    variance=1.0,
):
    """
    RBF kernel over input vectors.
    """
    if x2 is None:
        x2 = x1

    if x1.ndim == 1:
        x1 = x1.view(-1, 1)

    if x2.ndim == 1:
        x2 = x2.view(-1, 1)

    dists = torch.cdist(x1.float(), x2.float()).pow(2)

    return variance * torch.exp(-0.5 * dists / (lengthscale ** 2))


def full_gp_posterior_point(
    x_train,
    y_train,
    x_test,
    lengthscale=GP_LENGTHSCALE,
    variance=GP_VARIANCE,
    noise=GP_NOISE,
    jitter=1e-5,
):
    """
    Standard full GP posterior for point-valued scalar observations.

    Returns:
        posterior_mean: (N_test,)
        posterior_cov:  (N_test, N_test)
    """
    x_train = x_train.float()
    x_test = x_test.float()
    y_train = y_train.float().view(-1, 1)

    n_train = x_train.shape[0]
    n_test = x_test.shape[0]

    K = rbf_kernel(
        x_train,
        x_train,
        lengthscale=lengthscale,
        variance=variance,
    )

    K = K + noise * torch.eye(n_train, device=x_train.device)

    K_s = rbf_kernel(
        x_train,
        x_test,
        lengthscale=lengthscale,
        variance=variance,
    )

    K_ss = rbf_kernel(
        x_test,
        x_test,
        lengthscale=lengthscale,
        variance=variance,
    )

    K_ss = K_ss + jitter * torch.eye(n_test, device=x_test.device)

    current_jitter = jitter

    for _ in range(8):
        try:
            L = torch.linalg.cholesky(
                K + current_jitter * torch.eye(
                    n_train,
                    device=x_train.device,
                )
            )
            break
        except RuntimeError:
            current_jitter *= 10
    else:
        raise RuntimeError("Cholesky failed in full_gp_posterior_point.")

    alpha = torch.cholesky_solve(y_train, L)

    posterior_mean = K_s.t() @ alpha

    v = torch.cholesky_solve(K_s, L)
    posterior_cov = K_ss - K_s.t() @ v

    posterior_cov = 0.5 * (posterior_cov + posterior_cov.t())

    return posterior_mean.squeeze(-1), posterior_cov


def full_gp_posterior_heteroscedastic(
    x_train,
    y_train,
    x_test,
    noise_var_train,
    lengthscale=GP_LENGTHSCALE,
    variance=GP_VARIANCE,
    min_noise=GP_NOISE,
    jitter=1e-5,
):
    """
    Full GP posterior with heteroscedastic diagonal observation noise.
    Used for ITE-space IntervalGP head.
    """
    x_train = x_train.float()
    x_test = x_test.float()
    y_train = y_train.float().view(-1, 1)

    noise_var_train = noise_var_train.float().view(-1)
    noise_var_train = torch.clamp(noise_var_train, min=min_noise)

    n_train = x_train.shape[0]
    n_test = x_test.shape[0]

    K = rbf_kernel(
        x_train,
        x_train,
        lengthscale=lengthscale,
        variance=variance,
    )

    K = K + torch.diag(noise_var_train)

    K_s = rbf_kernel(
        x_train,
        x_test,
        lengthscale=lengthscale,
        variance=variance,
    )

    K_ss = rbf_kernel(
        x_test,
        x_test,
        lengthscale=lengthscale,
        variance=variance,
    )

    K_ss = K_ss + jitter * torch.eye(n_test, device=x_test.device)

    current_jitter = jitter

    for _ in range(8):
        try:
            L = torch.linalg.cholesky(
                K + current_jitter * torch.eye(
                    n_train,
                    device=x_train.device,
                )
            )
            break
        except RuntimeError:
            current_jitter *= 10
    else:
        raise RuntimeError("Cholesky failed in full_gp_posterior_heteroscedastic.")

    alpha = torch.cholesky_solve(y_train, L)

    posterior_mean = K_s.t() @ alpha

    v = torch.cholesky_solve(K_s, L)
    posterior_cov = K_ss - K_s.t() @ v

    posterior_cov = 0.5 * (posterior_cov + posterior_cov.t())

    return posterior_mean.squeeze(-1), posterior_cov


def latent_gp_posterior_smoothing(
    z_train,
    mu_u_train,
    z_test,
    lengthscale=GP_LENGTHSCALE,
    variance=GP_VARIANCE,
    noise=GP_NOISE,
):
    """
    Full GP posterior smoothing for latent confounder U.

    The training targets are encoder posterior means mu_u_train.
    Each latent dimension is smoothed independently.
    """
    if mu_u_train.ndim == 1:
        mu_u_train = mu_u_train.view(-1, 1)

    z_train_gp, mu_u_train_gp = maybe_subsample_gp_reference(
        z_train,
        mu_u_train,
        max_points=MAX_GP_POINTS,
        seed=123,
    )

    latent_dim = mu_u_train.shape[1]

    means = []
    vars_diag = []

    for r in range(latent_dim):
        mean_r, cov_r = full_gp_posterior_point(
            x_train=z_train_gp,
            y_train=mu_u_train_gp[:, r],
            x_test=z_test,
            lengthscale=lengthscale,
            variance=variance,
            noise=noise,
        )

        var_r = torch.diag(cov_r).clamp_min(1e-8)

        means.append(mean_r)
        vars_diag.append(var_r)

    mean = torch.stack(means, dim=1)
    std = torch.sqrt(torch.stack(vars_diag, dim=1))

    return mean, std


# %%
# ============================================================
# Cell 6: IntervalGP-VAE model consistent with the second code
# ============================================================

class DoubleEncoder(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dim,
        latent_dim,
        use_auxiliary_latents=False,
    ):
        super().__init__()

        self.use_aux = use_auxiliary_latents

        self.fc1 = nn.Linear(input_dim, hidden_dim)

        self.mean_u = nn.Linear(hidden_dim, latent_dim)
        self.logvar_u = nn.Linear(hidden_dim, latent_dim)

        self.mean_eps = nn.Linear(hidden_dim, input_dim)
        self.logvar_eps = nn.Linear(hidden_dim, input_dim)

        if self.use_aux:
            self.mean_ua0 = nn.Linear(hidden_dim, latent_dim)
            self.logvar_ua0 = nn.Linear(hidden_dim, latent_dim)

            self.mean_ua1 = nn.Linear(hidden_dim, latent_dim)
            self.logvar_ua1 = nn.Linear(hidden_dim, latent_dim)

    def forward(self, z):
        h = F.relu(self.fc1(z))

        mu_u = self.mean_u(h)
        logvar_u = self.logvar_u(h).clamp(-8.0, 4.0)
        std_u = torch.exp(0.5 * logvar_u)

        mu_eps = self.mean_eps(h)
        logvar_eps = self.logvar_eps(h).clamp(-8.0, 4.0)
        std_eps = torch.exp(0.5 * logvar_eps)

        if self.use_aux:
            mu_ua0 = self.mean_ua0(h)
            logvar_ua0 = self.logvar_ua0(h).clamp(-8.0, 4.0)
            std_ua0 = torch.exp(0.5 * logvar_ua0)

            mu_ua1 = self.mean_ua1(h)
            logvar_ua1 = self.logvar_ua1(h).clamp(-8.0, 4.0)
            std_ua1 = torch.exp(0.5 * logvar_ua1)

            return (
                mu_u,
                std_u,
                logvar_u,
                mu_eps,
                std_eps,
                logvar_eps,
                mu_ua0,
                std_ua0,
                logvar_ua0,
                mu_ua1,
                std_ua1,
                logvar_ua1,
            )

        return mu_u, std_u, logvar_u, mu_eps, std_eps, logvar_eps


class AdditiveDecoder(nn.Module):
    def __init__(
        self,
        latent_dim,
        hidden_dim,
        output_dim,
        use_auxiliary_latents=False,
    ):
        super().__init__()

        self.use_aux = use_auxiliary_latents

        factor = 1 + (2 if use_auxiliary_latents else 0)

        self.fc1 = nn.Linear(latent_dim * factor, hidden_dim)
        self.out = nn.Linear(hidden_dim, output_dim)

    def forward(self, u, eps, ua0=None, ua1=None):
        if self.use_aux:
            x = torch.cat([u, ua0, ua1], dim=1)
        else:
            x = u

        h = F.relu(self.fc1(x))

        return self.out(h) + eps


class GPVAEwithNoise(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dim,
        latent_dim,
        gp_lengthscale=GP_LENGTHSCALE,
        gp_variance=GP_VARIANCE,
        gp_noise=GP_NOISE,
        use_auxiliary_latents=False,
    ):
        super().__init__()

        self.use_aux = use_auxiliary_latents

        self.encoder = DoubleEncoder(
            input_dim=input_dim,
            hidden_dim=hidden_dim,
            latent_dim=latent_dim,
            use_auxiliary_latents=use_auxiliary_latents,
        )

        self.decoder = AdditiveDecoder(
            latent_dim=latent_dim,
            hidden_dim=hidden_dim,
            output_dim=input_dim,
            use_auxiliary_latents=use_auxiliary_latents,
        )

        self.gp_lengthscale = gp_lengthscale
        self.gp_variance = gp_variance
        self.gp_noise = gp_noise

    def reparameterize(self, mu, std):
        return mu + torch.randn_like(std) * std

    def local_interval_gp_regularizer(self, z, mu_u, std_u):
        """
        Local interval-valued GP prior regularizer.

        Training-time interval GP regularization term:
            log P(U in [mu_u - std_u, mu_u + std_u] | Z).

        This term is local and marginal.
        Full GP posterior smoothing is used at inference time.
        """
        k_ii = self.gp_variance * torch.ones_like(mu_u)
        gp_std = torch.sqrt(k_ii + self.gp_noise)

        lower = mu_u - std_u
        upper = mu_u + std_u

        normal = torch.distributions.Normal(
            loc=torch.zeros_like(mu_u),
            scale=gp_std,
        )

        prob = normal.cdf(upper) - normal.cdf(lower)
        prob = torch.clamp(prob, min=1e-8)

        return -torch.sum(torch.log(prob))

    def get_latent_stats(self, z, var_name="u"):
        outputs = self.encoder(z)

        if var_name == "u":
            return outputs[0], outputs[1], outputs[2]

        if var_name == "eps":
            return outputs[3], outputs[4], outputs[5]

        if var_name == "ua0":
            if not self.use_aux:
                return None, None, None
            return outputs[6], outputs[7], outputs[8]

        if var_name == "ua1":
            if not self.use_aux:
                return None, None, None
            return outputs[9], outputs[10], outputs[11]

        raise ValueError(f"Invalid var_name: {var_name}")

    def sample_latent(self, z, var_name="u"):
        mu, std, _ = self.get_latent_stats(z, var_name=var_name)

        if mu is None:
            return None

        return self.reparameterize(mu, std)

    def forward(self, z):
        if self.use_aux:
            (
                mu_u,
                std_u,
                logvar_u,
                mu_eps,
                std_eps,
                logvar_eps,
                mu_ua0,
                std_ua0,
                logvar_ua0,
                mu_ua1,
                std_ua1,
                logvar_ua1,
            ) = self.encoder(z)

            u = self.reparameterize(mu_u, std_u)
            eps = self.reparameterize(mu_eps, std_eps)
            ua0 = self.reparameterize(mu_ua0, std_ua0)
            ua1 = self.reparameterize(mu_ua1, std_ua1)

            z_recon = self.decoder(u, eps, ua0=ua0, ua1=ua1)

        else:
            (
                mu_u,
                std_u,
                logvar_u,
                mu_eps,
                std_eps,
                logvar_eps,
            ) = self.encoder(z)

            u = self.reparameterize(mu_u, std_u)
            eps = self.reparameterize(mu_eps, std_eps)

            ua0 = None
            ua1 = None

            z_recon = self.decoder(u, eps)

        # recon_loss = F.mse_loss(z_recon, z, reduction="sum")
        recon_loss = F.mse_loss(z_recon, z, reduction="sum") / z.shape[0] / z.shape[1]

        kl_u = -0.001 * torch.sum(
            1 + logvar_u - mu_u.pow(2) - logvar_u.exp()
        )

        kl_eps = -0.5 * torch.sum(
            1 + logvar_eps - mu_eps.pow(2) - logvar_eps.exp()
        )

        gp_interval_reg = self.local_interval_gp_regularizer(
            z=z,
            mu_u=mu_u,
            std_u=std_u,
        )

        std_penalty = torch.sum(std_u ** 2)

        total_loss = (
            recon_loss
            + kl_u
            + kl_eps
            + gp_interval_reg
            + 10.0 * std_penalty
        )

        if self.use_aux:
            kl_ua0 = -0.5 * torch.sum(
                1 + logvar_ua0 - mu_ua0.pow(2) - logvar_ua0.exp()
            )

            kl_ua1 = -0.5 * torch.sum(
                1 + logvar_ua1 - mu_ua1.pow(2) - logvar_ua1.exp()
            )

            total_loss = total_loss + kl_ua0 + kl_ua1

        return total_loss, {
            "recon_loss": recon_loss.item(),
            "kl_u": kl_u.item(),
            "kl_eps": kl_eps.item(),
            "gp_interval_reg": gp_interval_reg.item(),
            "u": u,
            "eps": eps,
            "ua0": ua0,
            "ua1": ua1,
            "mu_u": mu_u,
            "std_u": std_u,
            "z_recon": z_recon,
        }


class FlexibleCausalHead(nn.Module):
    def __init__(
        self,
        latent_dim_u,
        hidden_dim=CAUSAL_HEAD_HIDDEN_DIM,
        z_y_dim=None,
        use_auxiliary_latents=False,
    ):
        super().__init__()

        self.use_aux = use_auxiliary_latents
        self.z_y_dim = z_y_dim

        input_dim = latent_dim_u + 1

        if z_y_dim is not None:
            input_dim += z_y_dim

        if use_auxiliary_latents:
            input_dim += latent_dim_u

        self.fc = nn.Linear(input_dim, hidden_dim)
        self.out = nn.Linear(hidden_dim, 1)

    def forward(self, u, t, z_y=None, ua1=None):
        t = t.view(-1, 1)

        parts = [u, t]

        if self.z_y_dim is not None:
            if z_y is None:
                raise ValueError("Expected z_y input but got None.")
            parts.append(z_y)

        if self.use_aux:
            if ua1 is None:
                raise ValueError("Expected ua1 input but got None.")
            parts.append(ua1)

        x = torch.cat(parts, dim=1)

        h = F.relu(self.fc(x))

        return self.out(h)


def make_z_y(model, z):
    if model.causal_head.z_y_dim == z.shape[1]:
        return z

    if model.causal_head.z_y_dim:
        return z[:, z.shape[1] // 2:]

    return None


class CausalGPVAEwithNoise(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dim,
        latent_dim,
        z_y_dim=None,
        use_auxiliary_latents=False,
        gp_lengthscale=GP_LENGTHSCALE,
        gp_variance=GP_VARIANCE,
        gp_noise=GP_NOISE,
    ):
        super().__init__()

        self.z_y_dim = z_y_dim
        self.use_aux = use_auxiliary_latents

        self.vae = GPVAEwithNoise(
            input_dim=input_dim,
            hidden_dim=hidden_dim,
            latent_dim=latent_dim,
            gp_lengthscale=gp_lengthscale,
            gp_variance=gp_variance,
            gp_noise=gp_noise,
            use_auxiliary_latents=use_auxiliary_latents,
        )

        self.causal_head = FlexibleCausalHead(
            latent_dim_u=latent_dim,
            hidden_dim=CAUSAL_HEAD_HIDDEN_DIM,
            z_y_dim=z_y_dim,
            use_auxiliary_latents=use_auxiliary_latents,
        )

    def forward(self, z, t, y):
        loss_vae, vae_info = self.vae(z)

        u = vae_info["u"]
        ua1 = vae_info["ua1"] if self.use_aux else None

        z_y = make_z_y(self, z)

        y_pred = self.causal_head(
            u,
            t,
            z_y=z_y,
            ua1=ua1,
        )

        causal_loss = F.mse_loss(
            y_pred,
            y.unsqueeze(-1),
            reduction="sum",
        )

        total_loss = loss_vae + causal_loss

        return total_loss, {
            **vae_info,
            "y_pred": y_pred,
            "causal_loss": causal_loss.item(),
        }


# %%
# ============================================================
# Cell 7: ITE sampling and ITE-space IntervalGP posterior
# ============================================================

def sample_ite_from_encoder(
    model,
    z,
    n_samples=N_ITE_SAMPLES,
):
    """
    Sample ITE distribution using encoder posterior q(u|z) and q(ua1|z).

    Returns:
        ite_samples: (S, N)
        ite_mean:    (N,)
        ite_std:     (N,)
        ite_lower:   (N,)
        ite_upper:   (N,)
    """
    model.eval()

    with torch.no_grad():
        mu_u, std_u, _ = model.vae.get_latent_stats(z, var_name="u")

        N, d_u = mu_u.shape
        device = z.device

        z_y = make_z_y(model, z)

        mu_ua1, std_ua1, _ = model.vae.get_latent_stats(z, var_name="ua1")

        has_ua1 = (mu_ua1 is not None) and (std_ua1 is not None)

        eps_u = torch.randn(n_samples, N, d_u, device=device)
        u_samples = mu_u.unsqueeze(0) + eps_u * std_u.unsqueeze(0)

        U = u_samples.reshape(-1, d_u)

        if has_ua1:
            d_ua = mu_ua1.shape[1]
            eps_ua1 = torch.randn(n_samples, N, d_ua, device=device)
            ua1_samples = mu_ua1.unsqueeze(0) + eps_ua1 * std_ua1.unsqueeze(0)
            UA1 = ua1_samples.reshape(-1, d_ua)
        else:
            UA1 = None

        if z_y is not None:
            ZY = (
                z_y.unsqueeze(0)
                .expand(n_samples, -1, -1)
                .reshape(-1, z_y.shape[1])
            )
        else:
            ZY = None

        t0 = torch.zeros(U.shape[0], device=device)
        t1 = torch.ones(U.shape[0], device=device)

        y0 = model.causal_head(U, t0, z_y=ZY, ua1=UA1).squeeze(-1)
        y1 = model.causal_head(U, t1, z_y=ZY, ua1=UA1).squeeze(-1)

        ite_samples = (y1 - y0).view(n_samples, N)

        lower_q = (1.0 - ITE_CI) / 2.0
        upper_q = 1.0 - lower_q

        ite_mean = ite_samples.mean(dim=0)
        ite_std = ite_samples.std(dim=0)
        ite_lower = ite_samples.quantile(lower_q, dim=0)
        ite_upper = ite_samples.quantile(upper_q, dim=0)

        return ite_samples, ite_mean, ite_std, ite_lower, ite_upper


def ite_space_intervalgp_head(
    model,
    z_train,
    z_test,
    n_samples=N_ITE_SAMPLES,
    lengthscale=GP_LENGTHSCALE,
    variance=GP_VARIANCE,
    min_noise=GP_NOISE,
):
    """
    ITE-space IntervalGP head with full GP posterior.

    Steps:
      1. Use encoder posterior samples on training data to get interval-valued
         pseudo observations of ITE.
      2. Apply latent GP posterior smoothing:
             z -> u
      3. Use full GP posterior over lower, upper, and mean ITE functions:
             u -> tau(u)
      4. Return GP-smoothed ITE mean and interval for test data.
    """
    model.eval()

    with torch.no_grad():
        _, ite_mean_train, ite_std_train, ite_lower_train, ite_upper_train = (
            sample_ite_from_encoder(
                model=model,
                z=z_train,
                n_samples=n_samples,
            )
        )

        mu_u_train, _, _ = model.vae.get_latent_stats(z_train, var_name="u")
        mu_u_test, _, _ = model.vae.get_latent_stats(z_test, var_name="u")

        u_test_smooth, u_test_smooth_std = latent_gp_posterior_smoothing(
            z_train=z_train,
            mu_u_train=mu_u_train,
            z_test=z_test,
            lengthscale=lengthscale,
            variance=variance,
            noise=min_noise,
        )

        u_train_smooth = mu_u_train.detach()

        (
            u_train_gp,
            ite_lower_train_gp,
            ite_upper_train_gp,
            ite_mean_train_gp,
            ite_std_train_gp,
        ) = maybe_subsample_gp_reference(
            u_train_smooth,
            ite_lower_train.detach(),
            ite_upper_train.detach(),
            ite_mean_train.detach(),
            ite_std_train.detach(),
            max_points=MAX_GP_POINTS,
            seed=123,
        )

        noise_var_lower = ite_std_train_gp.pow(2) + min_noise
        noise_var_upper = ite_std_train_gp.pow(2) + min_noise
        noise_var_mean = ite_std_train_gp.pow(2) + min_noise

        gp_lower_mean, gp_lower_cov = full_gp_posterior_heteroscedastic(
            x_train=u_train_gp,
            y_train=ite_lower_train_gp,
            x_test=u_test_smooth,
            noise_var_train=noise_var_lower,
            lengthscale=lengthscale,
            variance=variance,
            min_noise=min_noise,
        )

        gp_upper_mean, gp_upper_cov = full_gp_posterior_heteroscedastic(
            x_train=u_train_gp,
            y_train=ite_upper_train_gp,
            x_test=u_test_smooth,
            noise_var_train=noise_var_upper,
            lengthscale=lengthscale,
            variance=variance,
            min_noise=min_noise,
        )

        gp_ite_mean, gp_ite_cov = full_gp_posterior_heteroscedastic(
            x_train=u_train_gp,
            y_train=ite_mean_train_gp,
            x_test=u_test_smooth,
            noise_var_train=noise_var_mean,
            lengthscale=lengthscale,
            variance=variance,
            min_noise=min_noise,
        )

        gp_lower_std = torch.sqrt(torch.diag(gp_lower_cov).clamp_min(1e-8))
        gp_upper_std = torch.sqrt(torch.diag(gp_upper_cov).clamp_min(1e-8))
        gp_ite_std = torch.sqrt(torch.diag(gp_ite_cov).clamp_min(1e-8))

        raw_lower = torch.minimum(gp_lower_mean, gp_upper_mean)
        raw_upper = torch.maximum(gp_lower_mean, gp_upper_mean)

        ite_lower_test = raw_lower - Z_VALUE_90 * gp_lower_std
        ite_upper_test = raw_upper + Z_VALUE_90 * gp_upper_std

        return {
            "ite_mean_test": gp_ite_mean,
            "ite_std_test": gp_ite_std,
            "ite_lower_test": ite_lower_test,
            "ite_upper_test": ite_upper_test,
            "u_test_smooth": u_test_smooth,
            "u_test_smooth_std": u_test_smooth_std,
            "u_test_raw": mu_u_test,
            "ite_lower_train": ite_lower_train,
            "ite_upper_train": ite_upper_train,
            "ite_mean_train": ite_mean_train,
            "ite_std_train": ite_std_train,
        }


# %%
# ============================================================
# Cell 8: Three-stage training
# ============================================================

def train_intervalgpvae(
    z_train,
    t_train,
    y_train,
    latent_dim,
    seed,
):
    """
    Three-stage training consistent with the second code.

    Stage 1:
        joint VAE + causal head training.

    Stage 2:
        freeze VAE and train causal head using posterior mean mu_u.

    Stage 3:
        freeze causal head and refine VAE with causal loss.
    """
    set_seed(seed)

    use_aux = chosen_version == "u_aux"

    if chosen_version == "z_to_y":
        z_y_dim = z_train.shape[1]
    elif chosen_version == "split_z_to_t_and_y":
        z_y_dim = z_train.shape[1] // 2
    else:
        z_y_dim = None

    dataset = TensorDataset(z_train, t_train, y_train)

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=False,
    )

    model = CausalGPVAEwithNoise(
        input_dim=z_train.shape[1],
        hidden_dim=INTERVALGPVAE_HIDDEN_DIM,
        latent_dim=latent_dim,
        z_y_dim=z_y_dim,
        use_auxiliary_latents=use_aux,
        gp_lengthscale=GP_LENGTHSCALE,
        gp_variance=GP_VARIANCE,
        gp_noise=GP_NOISE,
    ).to(DEVICE)

    # -----------------------------------------------------
    # Stage 1: joint training with AdamW
    # -----------------------------------------------------
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR_JOINT,
        weight_decay=WEIGHT_DECAY,
    )

    for epoch in range(JOINT_EPOCHS):
        model.train()

        for z_batch, t_batch, y_batch in loader:
            z_batch = z_batch.to(DEVICE)
            t_batch = t_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            loss, _ = model(z_batch, t_batch, y_batch)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

    # -----------------------------------------------------
    # Stage 2: train causal head with frozen VAE
    # -----------------------------------------------------
    for param in model.vae.parameters():
        param.requires_grad = False

    for param in model.causal_head.parameters():
        param.requires_grad = True

    causal_optimizer = torch.optim.AdamW(
        model.causal_head.parameters(),
        lr=LR_HEAD,
        weight_decay=WEIGHT_DECAY,
    )

    for epoch in range(HEAD_EPOCHS):
        model.train()

        for z_batch, t_batch, y_batch in loader:
            z_batch = z_batch.to(DEVICE)
            t_batch = t_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            with torch.no_grad():
                mu_u = model.vae.get_latent_stats(
                    z_batch,
                    var_name="u",
                )[0]

                if model.use_aux:
                    mu_ua1, std_ua1, _ = model.vae.get_latent_stats(
                        z_batch,
                        var_name="ua1",
                    )

                    ua1_batch = model.vae.reparameterize(
                        mu_ua1,
                        std_ua1,
                    )
                else:
                    ua1_batch = None

            z_y_batch = make_z_y(model, z_batch)

            y_pred = model.causal_head(
                mu_u.detach(),
                t_batch,
                z_y=z_y_batch,
                ua1=ua1_batch.detach() if ua1_batch is not None else None,
            )

            causal_loss = F.mse_loss(
                y_pred,
                y_batch.unsqueeze(-1),
            )

            causal_optimizer.zero_grad()
            causal_loss.backward()
            torch.nn.utils.clip_grad_norm_(
                model.causal_head.parameters(),
                max_norm=5.0,
            )
            causal_optimizer.step()

    # -----------------------------------------------------
    # Stage 3: refine VAE with frozen causal head
    # -----------------------------------------------------
    for param in model.causal_head.parameters():
        param.requires_grad = False

    for param in model.vae.parameters():
        param.requires_grad = True

    vae_optimizer = torch.optim.AdamW(
        model.vae.parameters(),
        lr=LR_VAE_REFINE,
        weight_decay=WEIGHT_DECAY,
    )

    for epoch in range(VAE_REFINE_EPOCHS):
        model.train()

        for z_batch, t_batch, y_batch in loader:
            z_batch = z_batch.to(DEVICE)
            t_batch = t_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            z_y_batch = make_z_y(model, z_batch)

            loss_vae, vae_info = model.vae(z_batch)

            ua1_batch = (
                model.vae.sample_latent(z_batch, var_name="ua1")
                if model.use_aux
                else None
            )

            y_pred = model.causal_head(
                vae_info["u"],
                t_batch,
                z_y=z_y_batch,
                ua1=ua1_batch,
            )

            causal_loss = F.mse_loss(
                y_pred,
                y_batch.unsqueeze(-1),
                reduction="sum",
            )

            total_loss = loss_vae + causal_loss

            vae_optimizer.zero_grad()
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(
                model.vae.parameters(),
                max_norm=5.0,
            )
            vae_optimizer.step()

    for param in model.parameters():
        param.requires_grad = True

    return model


# %%
# ============================================================
# Cell 9: Single experiment run
# ============================================================

@dataclass
class ExperimentResult:
    d: int
    k: int
    threshold: int
    above_threshold: bool
    n_train: int
    n_test: int
    seed: int
    pehe: float
    ate_error: float
    coverage: float
    interval_width: float
    latent_recovery_error_raw: float
    latent_recovery_error_smooth: float
    latent_abs_corr_raw: float
    latent_abs_corr_smooth: float
    runtime_sec: float


def run_single_experiment(d: int, k: int, seed: int) -> ExperimentResult:
    start = time.time()

    threshold = 2 * d + 4

    n_train = N_TRAIN_BY_D[d]
    n_test = N_TEST_BY_D[d]

    # --------------------------------------------------------
    # Generate train and test data with same structural directions
    # --------------------------------------------------------
    train_data = generate_synthetic_multidim(
        n=n_train,
        d=d,
        k=k,
        seed=seed,
        noise_z=NOISE_Z,
        noise_y=NOISE_Y,
    )

    test_data = generate_synthetic_multidim(
        n=n_test,
        d=d,
        k=k,
        seed=seed + 10000,
        noise_z=NOISE_Z,
        noise_y=NOISE_Y,
        proxy_directions=train_data.proxy_directions,
        proxy_params=train_data.proxy_params,
        treatment_direction=train_data.treatment_direction,
        tau_direction=train_data.tau_direction,
    )

    # --------------------------------------------------------
    # Standardize Z using training statistics
    # --------------------------------------------------------
    z_train, z_test, _, _ = standardize_train_test(
        train_data.z,
        test_data.z,
    )

    z_train = z_train.to(DEVICE)
    z_test = z_test.to(DEVICE)

    t_train = train_data.t.to(DEVICE)

    # --------------------------------------------------------
    # Standardize observed outcome Y for stable training
    # --------------------------------------------------------
    y_mean = train_data.y.mean()
    y_std = train_data.y.std().clamp_min(1e-6)

    y_train_std = ((train_data.y - y_mean) / y_std).to(DEVICE)

    # --------------------------------------------------------
    # Train IntervalGP-VAE
    # --------------------------------------------------------
    model = train_intervalgpvae(
        z_train=z_train,
        t_train=t_train,
        y_train=y_train_std,
        latent_dim=d,
        seed=seed,
    )

    model.eval()

    # --------------------------------------------------------
    # Inference:
    # latent GP smoothing + ITE-space GP posterior
    # --------------------------------------------------------
    intervalgp_results = ite_space_intervalgp_head(
        model=model,
        z_train=z_train,
        z_test=z_test,
        n_samples=N_ITE_SAMPLES,
        lengthscale=GP_LENGTHSCALE,
        variance=GP_VARIANCE,
        min_noise=GP_NOISE,
    )

    # --------------------------------------------------------
    # Convert standardized-scale ITE back to original Y scale
    # --------------------------------------------------------
    ite_mean_std = intervalgp_results["ite_mean_test"]
    ite_lower_std = intervalgp_results["ite_lower_test"]
    ite_upper_std = intervalgp_results["ite_upper_test"]

    y_std_device = y_std.to(DEVICE)

    ite_mean = ite_mean_std * y_std_device
    ite_lower = ite_lower_std * y_std_device
    ite_upper = ite_upper_std * y_std_device

    true_ite = test_data.ite

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------
    pehe_value = pehe_np(
        est_ite=ite_mean,
        true_ite=true_ite,
    )

    ate_est = float(to_numpy_1d(ite_mean).mean())
    ate_true = float(to_numpy_1d(true_ite).mean())
    ate_error = abs(ate_est - ate_true)

    interval_info = compute_interval_metrics(
        lower=ite_lower,
        upper=ite_upper,
        true_value=true_ite,
    )

    u_raw = intervalgp_results["u_test_raw"].detach().cpu().numpy()
    u_smooth = intervalgp_results["u_test_smooth"].detach().cpu().numpy()
    u_true = test_data.u.detach().cpu().numpy()

    latent_err_raw = latent_recovery_error_linear_aligned(
        u_true=u_true,
        u_hat=u_raw,
    )

    latent_err_smooth = latent_recovery_error_linear_aligned(
        u_true=u_true,
        u_hat=u_smooth,
    )

    latent_corr_raw = latent_abs_corr_mean(
        u_true=u_true,
        u_hat=u_raw,
    )

    latent_corr_smooth = latent_abs_corr_mean(
        u_true=u_true,
        u_hat=u_smooth,
    )

    runtime = time.time() - start

    del model
    reset_memory()

    return ExperimentResult(
        d=d,
        k=k,
        threshold=threshold,
        above_threshold=(k >= threshold),
        n_train=n_train,
        n_test=n_test,
        seed=seed,
        pehe=pehe_value,
        ate_error=ate_error,
        coverage=interval_info["coverage"],
        interval_width=interval_info["avg_length"],
        latent_recovery_error_raw=latent_err_raw,
        latent_recovery_error_smooth=latent_err_smooth,
        latent_abs_corr_raw=latent_corr_raw,
        latent_abs_corr_smooth=latent_corr_smooth,
        runtime_sec=runtime,
    )


# %%
# ============================================================
# Cell 10: Full grid experiment
# ============================================================

all_results = []

total_runs = sum(
    len(K_GRID_BY_D[d]) * len(SEEDS)
    for d in D_LIST
)

run_id = 0

for d in D_LIST:
    for k in K_GRID_BY_D[d]:
        for seed in SEEDS:
            run_id += 1

            print("\n" + "=" * 100)
            print(
                f"Run {run_id}/{total_runs}: "
                f"d={d}, k={k}, threshold={2*d+4}, "
                f"n_train={N_TRAIN_BY_D[d]}, n_test={N_TEST_BY_D[d]}, seed={seed}"
            )
            print("=" * 100)

            try:
                result = run_single_experiment(
                    d=d,
                    k=k,
                    seed=seed,
                )

                row = asdict(result)
                all_results.append(row)

                print(
                    f"Done | "
                    f"PEHE={result.pehe:.4f}, "
                    f"ATE error={result.ate_error:.4f}, "
                    f"coverage={result.coverage:.4f}, "
                    f"width={result.interval_width:.4f}, "
                    f"latent corr smooth={result.latent_abs_corr_smooth:.4f}, "
                    f"time={result.runtime_sec:.2f}s"
                )

            except Exception as e:
                print(f"Error at d={d}, k={k}, seed={seed}: {e}")

                all_results.append(
                    {
                        "d": d,
                        "k": k,
                        "threshold": 2 * d + 4,
                        "above_threshold": k >= 2 * d + 4,
                        "n_train": N_TRAIN_BY_D[d],
                        "n_test": N_TEST_BY_D[d],
                        "seed": seed,
                        "error": str(e),
                    }
                )

            partial_df = pd.DataFrame(all_results)

            partial_path = os.path.join(
                OUTPUT_DIR,
                "experiment_c_consistent_intervalgpvae_partial_results.csv",
            )

            partial_df.to_csv(partial_path, index=False)

            print(f"Partial results saved to: {partial_path}")


# %%
# ============================================================
# Cell 11: Save raw and summary results
# ============================================================

raw_df = pd.DataFrame(all_results)

raw_path = os.path.join(
    OUTPUT_DIR,
    "experiment_c_consistent_intervalgpvae_results_raw.csv",
)

raw_df.to_csv(raw_path, index=False)

print("\n" + "=" * 100)
print(f"Raw results saved to: {raw_path}")
print("=" * 100)

display(raw_df)


valid_df = raw_df[raw_df["pehe"].notna()].copy()

summary_df = (
    valid_df
    .groupby(
        [
            "d",
            "k",
            "threshold",
            "above_threshold",
            "n_train",
            "n_test",
        ],
        dropna=False,
    )
    .agg(
        num_runs=("pehe", "count"),
        pehe_mean=("pehe", "mean"),
        pehe_std=("pehe", "std"),
        ate_error_mean=("ate_error", "mean"),
        ate_error_std=("ate_error", "std"),
        coverage_mean=("coverage", "mean"),
        coverage_std=("coverage", "std"),
        interval_width_mean=("interval_width", "mean"),
        interval_width_std=("interval_width", "std"),
        latent_recovery_error_raw_mean=("latent_recovery_error_raw", "mean"),
        latent_recovery_error_raw_std=("latent_recovery_error_raw", "std"),
        latent_recovery_error_smooth_mean=("latent_recovery_error_smooth", "mean"),
        latent_recovery_error_smooth_std=("latent_recovery_error_smooth", "std"),
        latent_abs_corr_raw_mean=("latent_abs_corr_raw", "mean"),
        latent_abs_corr_raw_std=("latent_abs_corr_raw", "std"),
        latent_abs_corr_smooth_mean=("latent_abs_corr_smooth", "mean"),
        latent_abs_corr_smooth_std=("latent_abs_corr_smooth", "std"),
        runtime_sec_mean=("runtime_sec", "mean"),
        runtime_sec_std=("runtime_sec", "std"),
    )
    .reset_index()
)

summary_path = os.path.join(
    OUTPUT_DIR,
    "experiment_c_consistent_intervalgpvae_results_summary.csv",
)

summary_df.to_csv(summary_path, index=False)

print("\n" + "=" * 100)
print(f"Summary results saved to: {summary_path}")
print("=" * 100)

display(summary_df)


# %%
# ============================================================
# Cell 12: Compact paper-style summary
# ============================================================

compact_summary = summary_df[
    [
        "d",
        "k",
        "threshold",
        "above_threshold",
        "n_train",
        "n_test",
        "num_runs",
        "pehe_mean",
        "pehe_std",
        "ate_error_mean",
        "ate_error_std",
        "coverage_mean",
        "coverage_std",
        "interval_width_mean",
        "interval_width_std",
        "latent_recovery_error_smooth_mean",
        "latent_recovery_error_smooth_std",
        "latent_abs_corr_smooth_mean",
        "latent_abs_corr_smooth_std",
        "runtime_sec_mean",
        "runtime_sec_std",
    ]
].copy()

compact_path = os.path.join(
    OUTPUT_DIR,
    "experiment_c_consistent_intervalgpvae_compact_summary.csv",
)

compact_summary.to_csv(compact_path, index=False)

print("\nCompact summary:")
display(compact_summary)

print(f"\nCompact summary saved to: {compact_path}")


# %%
# ============================================================
# Cell 13: Threshold-level summary
# ============================================================

threshold_summary = (
    valid_df
    .groupby(["d", "above_threshold"], dropna=False)
    .agg(
        num_runs=("pehe", "count"),
        n_train=("n_train", "first"),
        n_test=("n_test", "first"),
        pehe_mean=("pehe", "mean"),
        pehe_std=("pehe", "std"),
        ate_error_mean=("ate_error", "mean"),
        ate_error_std=("ate_error", "std"),
        coverage_mean=("coverage", "mean"),
        coverage_std=("coverage", "std"),
        interval_width_mean=("interval_width", "mean"),
        interval_width_std=("interval_width", "std"),
        latent_recovery_error_smooth_mean=("latent_recovery_error_smooth", "mean"),
        latent_recovery_error_smooth_std=("latent_recovery_error_smooth", "std"),
        latent_abs_corr_smooth_mean=("latent_abs_corr_smooth", "mean"),
        latent_abs_corr_smooth_std=("latent_abs_corr_smooth", "std"),
        runtime_sec_mean=("runtime_sec", "mean"),
        runtime_sec_std=("runtime_sec", "std"),
    )
    .reset_index()
)

threshold_path = os.path.join(
    OUTPUT_DIR,
    "experiment_c_consistent_intervalgpvae_threshold_summary.csv",
)

threshold_summary.to_csv(threshold_path, index=False)

print("\nThreshold-level summary:")
display(threshold_summary)

print(f"\nThreshold-level summary saved to: {threshold_path}")




# End

In [ ]:
# %%
# ============================================================
# Cell 14: Optional heatmap plotting
# Mark out cases where k = 2d + 4
# Plot y-axis as d = 1,2,3,4,5 from top to bottom
# ============================================================

def make_metric_heatmap(
    summary_df,
    metric_col,
    title,
    filename,
    cmap="viridis",
    mark_threshold=True,
):
    """
    Draw heatmap for one metric.

    Requirements:
    1. Mark cells where k = 2d + 4.
    2. Plot y-axis from d = 1,2,3,4,5 from top to bottom.
    """
    pivot = summary_df.pivot(
        index="d",
        columns="k",
        values=metric_col,
    )

    # --------------------------------------------------------
    # Ensure d is ordered as 1,2,3,4,5 from top to bottom
    # --------------------------------------------------------
    pivot = pivot.sort_index(ascending=True)
    pivot = pivot.reindex(sorted(pivot.columns), axis=1)

    d_values = list(pivot.index)
    k_values = list(pivot.columns)

    plt.figure(figsize=(10, 5))

    im = plt.imshow(
        pivot.values,
        aspect="auto",
        cmap=cmap,
        origin="upper",   # important: first row d=1 appears at the top
    )

    plt.colorbar(im, label=metric_col)

    plt.xticks(
        ticks=np.arange(len(k_values)),
        labels=k_values,
    )

    plt.yticks(
        ticks=np.arange(len(d_values)),
        labels=d_values,
    )

    plt.xlabel("Number of proxies k")
    plt.ylabel("Latent dimension d")
    plt.title(title)

    # --------------------------------------------------------
    # Write metric value inside each cell
    # --------------------------------------------------------
    for i, d_val in enumerate(d_values):
        for j, k_val in enumerate(k_values):
            value = pivot.loc[d_val, k_val]

            if np.isfinite(value):
                plt.text(
                    j,
                    i,
                    f"{value:.3f}",
                    ha="center",
                    va="center",
                    color="white",
                    fontsize=8,
                )

    # --------------------------------------------------------
    # Mark threshold cells: k = 2d + 4
    # --------------------------------------------------------
    if mark_threshold:
        ax = plt.gca()

        for i, d_val in enumerate(d_values):
            threshold_k = 2 * int(d_val) + 4

            if threshold_k in k_values:
                j = k_values.index(threshold_k)

                rect = plt.Rectangle(
                    (j - 0.5, i - 0.5),
                    1.0,
                    1.0,
                    fill=False,
                    edgecolor="red",
                    linewidth=2.5,
                )

                ax.add_patch(rect)

                plt.text(
                    j,
                    i + 0.32,
                    r"$k=2d+4$",
                    ha="center",
                    va="center",
                    color="red",
                    fontsize=7,
                    fontweight="bold",
                )

    plt.tight_layout()

    path = os.path.join(OUTPUT_DIR, filename)

    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"Saved heatmap to: {path}")


if MAKE_HEATMAPS:
    make_metric_heatmap(
        summary_df=summary_df,
        metric_col="pehe_mean",
        title="PEHE across latent dimension d and proxy count k",
        filename="heatmap_pehe_consistent_intervalgpvae_marked_threshold.png",
        cmap="viridis",
        mark_threshold=True,
    )

    make_metric_heatmap(
        summary_df=summary_df,
        metric_col="coverage_mean",
        title="ITE interval coverage across latent dimension d and proxy count k",
        filename="heatmap_coverage_consistent_intervalgpvae_marked_threshold.png",
        cmap="plasma",
        mark_threshold=True,
    )

    make_metric_heatmap(
        summary_df=summary_df,
        metric_col="interval_width_mean",
        title="Mean ITE interval width across latent dimension d and proxy count k",
        filename="heatmap_interval_width_consistent_intervalgpvae_marked_threshold.png",
        cmap="magma",
        mark_threshold=True,
    )

    make_metric_heatmap(
        summary_df=summary_df,
        metric_col="latent_recovery_error_smooth_mean",
        title="Smoothed latent recovery error across d and k",
        filename="heatmap_latent_recovery_error_smooth_consistent_intervalgpvae_marked_threshold.png",
        cmap="inferno",
        mark_threshold=True,
    )

    make_metric_heatmap(
        summary_df=summary_df,
        metric_col="latent_abs_corr_smooth_mean",
        title="Smoothed latent absolute correlation across d and k",
        filename="heatmap_latent_abs_corr_smooth_consistent_intervalgpvae_marked_threshold.png",
        cmap="cividis",
        mark_threshold=True,
    )